In [1]:
# ================================================================================
# BLOQUE 1: IMPORTS Y SETUP BÁSICO
# ================================================================================

# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import pickle
import gc
from pathlib import Path
from datetime import datetime

# Procesamiento de imágenes
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# PyTorch Vision
import torchvision.transforms as transforms
import torchvision.models as models

# ================================================================================
# FUNCIÓN SETUP ENVIRONMENT
# ================================================================================

def setup_environment(seed=42):
    """Configurar reproducibilidad y device"""
    # Reproducibilidad
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Limpiar GPU si está disponible
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        gc.collect()
        print(f"🔥 GPU detectada: {torch.cuda.get_device_name(0)}")
        print(f"   Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    else:
        print("⚠️ GPU NO DISPONIBLE - El entrenamiento será muy lento")
    
    return device

# ================================================================================
# VERIFICACIÓN SIMPLE DE GPU
# ================================================================================

def verificar_gpu_simple():
    """Verificación simple y directa de GPU"""
    print("🔥 VERIFICACIÓN SIMPLE DE GPU")
    print("=" * 30)
    
    if torch.cuda.is_available():
        print("✅ GPU disponible")
        
        gpu_name = torch.cuda.get_device_name(0)
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        print(f"   GPU: {gpu_name}")
        print(f"   Memoria: {total_memory:.1f} GB")
        
        print("🧪 Test rápido...")
        test_tensor = torch.randn(100, 100).cuda()
        result = test_tensor @ test_tensor
        print("✅ GPU funciona correctamente")
        
        del test_tensor, result
        torch.cuda.empty_cache()
        
        return True
        
    else:
        print("❌ GPU NO disponible")
        print("💡 Usando CPU (será muy lento)")
        return False

# EJECUTAR SETUP
print("🚀 CONFIGURACIÓN INICIAL")
print("=" * 25)

# Configurar environment
device = setup_environment()

# Verificar GPU
gpu_ok = verificar_gpu_simple()

print(f"\n🎯 Device configurado: {device}")
print(f"📊 Estado GPU: {'OK' if gpu_ok else 'NO DISPONIBLE'}")

if gpu_ok:
    print("🚀 ¡Listo para continuar!")
else:
    print("⚠️ Continuar con CPU será MUY lento")

🚀 CONFIGURACIÓN INICIAL
🔥 GPU detectada: NVIDIA GeForce RTX 3050 Laptop GPU
   Memoria total: 4.0 GB
🔥 VERIFICACIÓN SIMPLE DE GPU
✅ GPU disponible
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   Memoria: 4.0 GB
🧪 Test rápido...
✅ GPU funciona correctamente

🎯 Device configurado: cuda
📊 Estado GPU: OK
🚀 ¡Listo para continuar!


In [2]:
# ================================================================================
# BLOQUE 2: DESCARGAR DATASET TINY-IMAGENET-200
# ================================================================================
import urllib.request
import zipfile
from pathlib import Path

project_root = Path().resolve()
dataset_dir  = project_root / "dataset"
zip_path     = dataset_dir / "tiny-imagenet-200.zip"
extracted    = dataset_dir / "tiny-imagenet-200"

if extracted.exists() and (extracted / "train").exists():
    print(f"✅ Dataset ya existe en: {extracted}")
else:
    dataset_dir.mkdir(exist_ok=True)

    download_url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    print(f"📥 Descargando Tiny ImageNet-200 (~237 MB)...")
    print(f"   URL: {download_url}")

    if not zip_path.exists():
        def _progress(count, block_size, total_size):
            if total_size > 0 and count % 500 == 0:
                pct = min(count * block_size / total_size * 100, 100)
                print(f"   {pct:.1f}%", end="\r")

        urllib.request.urlretrieve(download_url, zip_path, reporthook=_progress)
        print(f"\n   ✅ Descarga completada: {zip_path.stat().st_size / 1e6:.1f} MB")

    print(f"\n📦 Extrayendo en {dataset_dir} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dataset_dir)
    print("   ✅ Extracción completada")

    zip_path.unlink()
    print("   🗑️ ZIP eliminado")

print(f"\n🎯 Dataset listo en: {extracted}")

📥 Descargando Tiny ImageNet-200 (~237 MB)...
   URL: http://cs231n.stanford.edu/tiny-imagenet-200.zip
   99.1%
   ✅ Descarga completada: 248.1 MB

📦 Extrayendo en C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\dataset ...
   ✅ Extracción completada
   🗑️ ZIP eliminado

🎯 Dataset listo en: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\dataset\tiny-imagenet-200


In [3]:
# ================================================================================
# BLOQUE 3: CONFIGURAR PATHS
# ================================================================================
from pathlib import Path

project_root = Path().resolve()

paths = {
    'base':           project_root,
    'train_csv':      project_root / "train_data.csv",
    'val_csv':        project_root / "val_data.csv",
    'test_csv':       project_root / "test_data.csv",
    'dataset_folder': project_root / "dataset" / "tiny-imagenet-200",
    'mappings':       project_root / "path_mappings.pkl",
    'checkpoints':    project_root / "checkpoints",
}

paths['checkpoints'].mkdir(exist_ok=True)

print("📁 PATHS CONFIGURADOS")
print("=" * 25)
for key, val in paths.items():
    print(f"   {key}: {val}")
print("✅ Directorio checkpoints creado/verificado")

📁 PATHS CONFIGURADOS
   base: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet
   train_csv: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\train_data.csv
   val_csv: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\val_data.csv
   test_csv: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\test_data.csv
   dataset_folder: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\dataset\tiny-imagenet-200
   mappings: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\path_mappings.pkl
   checkpoints: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\checkpoints
✅ Directorio checkpoints creado/verificado


In [4]:
# ================================================================================
# BLOQUE 3: CREAR MAPPINGS DE ARCHIVOS
# ================================================================================

def limpiar_mappings_antiguos(paths):
    """Eliminar mappings antiguos con rutas incorrectas"""
    print("🧹 LIMPIANDO MAPPINGS ANTIGUOS")
    print("=" * 30)
    
    mappings_file = paths['mappings']
    
    if mappings_file.exists():
        print("   🗑️ Eliminando path_mappings.pkl antiguo...")
        mappings_file.unlink()
        print("   ✅ Mappings antiguos eliminados")
    else:
        print("   ✅ No hay mappings antiguos que eliminar")

def verificar_estructura_dataset(paths):
    """Verificar y mostrar la estructura real del dataset"""
    print("\n🔍 VERIFICANDO ESTRUCTURA REAL DEL DATASET")
    print("=" * 45)
    
    ubicaciones_posibles = [
        paths['dataset_folder'],                             # dataset/tiny-imagenet-200 (primary)
        paths['base'] / "dataset" / "tiny-imagenet-200",    # explicit fallback
        paths['base'] / "tiny-imagenet-200",                # if user placed it manually at root
    ]
    
    print("📁 Buscando dataset en:")
    for ubicacion in ubicaciones_posibles:
        print(f"   📂 {ubicacion}")
        if ubicacion.exists():
            print(f"      ✅ Carpeta existe")
            
            train_folder = ubicacion / "train"
            val_folder = ubicacion / "val"
            
            if train_folder.exists() and val_folder.exists():
                print(f"      ✅ train/ y val/ encontradas")
                val_images = val_folder / "images"
                if val_images.exists():
                    print(f"      ✅ val/images/ encontrada")
                    try:
                        num_files = len(list(val_images.glob("*.JPEG")))
                        print(f"      📊 {num_files} archivos .JPEG en val/images/")
                    except:
                        print(f"      ⚠️ Error contando archivos")
                print(f"      🎯 ¡UBICACIÓN CORRECTA ENCONTRADA!")
                return ubicacion
        else:
            print(f"      ❌ Carpeta no existe")
    
    print("\n❌ No se encontró estructura válida del dataset")
    return None

def crear_mappings_corregidos(paths, dataset_path):
    """Crear mappings desde cero con la ruta correcta"""
    print(f"\n🗺️ CREANDO MAPPINGS CORREGIDOS")
    print("=" * 35)
    print(f"📍 Dataset ubicado en: {dataset_path}")
    
    mappings = {'train': {}, 'val': {}, 'test': {}}
    
    # TRAIN MAPPINGS
    print("\n📈 Mapeando archivos de train...")
    train_df = pd.read_csv(paths['train_csv'])
    train_encontrados = 0
    
    for idx, row in train_df.iterrows():
        kaggle_path = row['File']
        relative_path = kaggle_path.replace('/kaggle/input/tiny-imagenet/tiny-imagenet-200/', '')
        local_path = dataset_path / relative_path
        
        if local_path.exists():
            mappings['train'][kaggle_path] = str(local_path)
            train_encontrados += 1
        
        if idx % 10000 == 0 and idx > 0:
            print(f"      Progreso: {idx:,} - {train_encontrados:,} encontrados")
    
    print(f"   ✅ Train: {train_encontrados:,}/{len(train_df):,} mapeados")
    
    # VAL MAPPINGS  
    print("\n📊 Mapeando archivos de val...")
    val_df = pd.read_csv(paths['val_csv'])
    val_encontrados = 0
    
    for idx, row in val_df.iterrows():
        filename = row['File']
        local_path = dataset_path / "val" / "images" / filename
        
        if local_path.exists():
            mappings['val'][filename] = str(local_path)
            val_encontrados += 1
    
    print(f"   ✅ Val: {val_encontrados:,}/{len(val_df):,} mapeados")
    
    # TEST MAPPINGS
    print("\n🧪 Mapeando archivos de test...")
    test_df = pd.read_csv(paths['test_csv'])
    test_encontrados = 0
    
    for idx, row in test_df.iterrows():
        filename = row['File']
        local_path = dataset_path / "val" / "images" / filename
        
        if local_path.exists():
            mappings['test'][filename] = str(local_path)
            test_encontrados += 1
    
    print(f"   ✅ Test: {test_encontrados:,}/{len(test_df):,} mapeados")
    
    # GUARDAR MAPPINGS
    total_encontrado = train_encontrados + val_encontrados + test_encontrados
    total_esperado = len(train_df) + len(val_df) + len(test_df)
    
    print(f"\n📊 RESUMEN FINAL:")
    print(f"   Total mapeado: {total_encontrado:,}/{total_esperado:,} ({total_encontrado/total_esperado*100:.1f}%)")
    
    if total_encontrado > 0:
        mappings_file = paths['mappings']
        with open(mappings_file, 'wb') as f:
            pickle.dump(mappings, f)
        print(f"   💾 Mappings guardados en: {mappings_file}")
        return mappings
    else:
        print("   ❌ No se pudieron crear mappings")
        return None

def verificar_mappings_nuevos(mappings, paths):
    """Verificar que los nuevos mappings funcionen"""
    print(f"\n🔍 VERIFICANDO MAPPINGS NUEVOS")
    print("=" * 30)
    
    val_df = pd.read_csv(paths['val_csv'])
    sample_files = val_df.head(5)
    
    exitos = 0
    for _, row in sample_files.iterrows():
        filename = row['File']
        label = row['Encoded_Label']
        
        if filename in mappings['val']:
            ruta_real = mappings['val'][filename]
            try:
                img = Image.open(ruta_real)
                print(f"   ✅ {filename} -> Clase {label} -> {img.size}")
                exitos += 1
            except Exception as e:
                print(f"   ❌ Error: {e}")
        else:
            print(f"   ❌ {filename} no mapeado")
    
    print(f"\n📊 Resultado: {exitos}/5 imágenes verificadas")
    return exitos >= 3

def ejecutar_paso3_mappings(paths):
    """Ejecutar paso 3 completo"""
    print("🗺️ PASO 3: CREAR MAPPINGS")
    print("=" * 25)
    
    # Limpiar mappings antiguos
    limpiar_mappings_antiguos(paths)
    
    # Verificar estructura real del dataset
    dataset_path = verificar_estructura_dataset(paths)
    
    if dataset_path is None:
        print("❌ No se puede proceder sin encontrar el dataset")
        return None
    
    # Crear mappings nuevos
    mappings = crear_mappings_corregidos(paths, dataset_path)
    
    if mappings is None:
        print("❌ Error creando mappings")
        return None
    
    # Verificar que funcionen
    if verificar_mappings_nuevos(mappings, paths):
        print("\n🎯 ¡MAPPINGS CREADOS EXITOSAMENTE!")
        return mappings
    else:
        print("\n❌ Los mappings tienen problemas")
        return None

# EJECUTAR PASO 3
print("🗺️ EJECUTANDO PASO 3: MAPPINGS")
print("=" * 35)

mappings = ejecutar_paso3_mappings(paths)

if mappings:
    print(f"\n✅ PASO 3 COMPLETADO")
    print(f"📊 Mappings:")
    print(f"   Train: {len(mappings['train']):,}")
    print(f"   Val: {len(mappings['val']):,}")
    print(f"   Test: {len(mappings['test']):,}")
else:
    print(f"\n❌ PASO 3 FALLÓ")

🗺️ EJECUTANDO PASO 3: MAPPINGS
🗺️ PASO 3: CREAR MAPPINGS
🧹 LIMPIANDO MAPPINGS ANTIGUOS
   ✅ No hay mappings antiguos que eliminar

🔍 VERIFICANDO ESTRUCTURA REAL DEL DATASET
📁 Buscando dataset en:
   📂 C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\dataset\tiny-imagenet-200
      ✅ Carpeta existe
      ✅ train/ y val/ encontradas
      ✅ val/images/ encontrada
      📊 10000 archivos .JPEG en val/images/
      🎯 ¡UBICACIÓN CORRECTA ENCONTRADA!

🗺️ CREANDO MAPPINGS CORREGIDOS
📍 Dataset ubicado en: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\dataset\tiny-imagenet-200

📈 Mapeando archivos de train...
      Progreso: 10,000 - 10,001 encontrados
      Progreso: 20,000 - 20,001 encontrados
      Progreso: 30,000 - 30,001 encontrados
      Progreso: 40,000 - 40,001 encontrados
      Progreso: 50,000 - 50,001 encontrados
      Progreso: 60,000 - 60,001 encontrados
      Progreso: 70,0

In [5]:
# ================================================================================
# BLOQUE 4: DATALOADERS Y DATASET PERSONALIZADO
# ================================================================================

class TinyImageNetDataset(Dataset):
    """Dataset personalizado para Tiny ImageNet"""
    
    def __init__(self, csv_file, mappings, transform=None):
        self.df = pd.read_csv(csv_file)
        self.mappings = mappings
        self.transform = transform
        print(f"📊 Dataset creado: {len(self.df):,} imágenes")
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['File']
        
        if filename not in self.mappings:
            raise KeyError(f"Archivo {filename} no encontrado en mappings")
        
        real_path = self.mappings[filename]
        
        try:
            image = Image.open(real_path).convert('RGB')
        except Exception as e:
            raise IOError(f"Error cargando {real_path}: {e}")
        
        if self.transform:
            image = self.transform(image)
            
        label = row['Encoded_Label']
        return image, label

class TestDataset(Dataset):
    """Dataset para test (sin labels)"""
    
    def __init__(self, csv_file, mappings, transform=None):
        self.df = pd.read_csv(csv_file)
        self.mappings = mappings
        self.transform = transform
        print(f"🧪 Test Dataset: {len(self.df):,} imágenes")
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['File']
        
        if filename not in self.mappings:
            raise KeyError(f"Archivo {filename} no encontrado en mappings")
        
        real_path = self.mappings[filename]
        
        try:
            image = Image.open(real_path).convert('RGB')
        except Exception as e:
            raise IOError(f"Error cargando {real_path}: {e}")
        
        if self.transform:
            image = self.transform(image)
            
        return image, filename

def obtener_transformaciones(tipo='conservative'):
    """Obtener transformaciones según el tipo de experimento"""
    
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
    
    if tipo == 'basic':
        train_transform = transforms.Compose([
            transforms.ToTensor(),
            normalize
        ])
        
    elif tipo == 'conservative':
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=8),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.RandomCrop(64, padding=4, padding_mode='reflect'),
            transforms.ToTensor(),
            normalize
        ])
        
    elif tipo == 'aggressive':
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
            transforms.ToTensor(),
            normalize
        ])
    
    else:
        raise ValueError(f"Tipo '{tipo}' no reconocido")
    
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        normalize
    ])
    
    return train_transform, val_transform

def crear_dataloaders(paths, mappings, batch_size=32, tipo_transforms='conservative'):
    """Crear DataLoaders para train, val y test"""
    
    print("🚀 CREANDO DATALOADERS")
    print("=" * 25)
    
    # Obtener transformaciones
    print(f"🔧 Configurando transformaciones: {tipo_transforms}")
    train_transform, val_transform = obtener_transformaciones(tipo_transforms)
    
    # Crear datasets
    print("📊 Creando datasets...")
    
    train_dataset = TinyImageNetDataset(
        paths['train_csv'], 
        mappings['train'], 
        train_transform
    )
    
    val_dataset = TinyImageNetDataset(
        paths['val_csv'], 
        mappings['val'], 
        val_transform
    )
    
    test_dataset = TestDataset(
        paths['test_csv'], 
        mappings['test'], 
        val_transform
    )
    
    # Configuración para DataLoaders
    num_workers = 0
    pin_memory = torch.cuda.is_available()
    
    print(f"⚙️ Configuración DataLoaders:")
    print(f"   Batch size: {batch_size}")
    print(f"   Num workers: {num_workers}")
    print(f"   Pin memory: {pin_memory}")
    
    # Crear DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    
    print(f"\n📊 DataLoaders creados:")
    print(f"   Train: {len(train_dataset):,} imágenes, {len(train_loader):,} batches")
    print(f"   Val: {len(val_dataset):,} imágenes, {len(val_loader):,} batches")
    print(f"   Test: {len(test_dataset):,} imágenes, {len(test_loader):,} batches")
    
    return train_loader, val_loader, test_loader

def verificar_dataloaders(train_loader, val_loader, test_loader, device):
    """Verificar que los DataLoaders funcionen correctamente"""
    print(f"\n🔍 VERIFICANDO DATALOADERS")
    print("=" * 25)
    
    try:
        # Probar cada DataLoader
        train_batch = next(iter(train_loader))
        train_images, train_labels = train_batch
        
        val_batch = next(iter(val_loader))
        val_images, val_labels = val_batch
        
        test_batch = next(iter(test_loader))
        test_images, test_filenames = test_batch
        
        print(f"📏 Shapes:")
        print(f"   Train: {train_images.shape} (imágenes), {train_labels.shape} (labels)")
        print(f"   Val: {val_images.shape} (imágenes), {val_labels.shape} (labels)")
        print(f"   Test: {test_images.shape} (imágenes)")
        
        # Verificar transferencia a GPU
        print(f"\n🔥 Probando transferencia a {device}...")
        train_images_gpu = train_images.to(device)
        train_labels_gpu = train_labels.to(device)
        print(f"   ✅ Transferencia exitosa")
        
        # Limpiar memoria
        del train_images_gpu, train_labels_gpu
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        
        print(f"\n🎉 ¡DATALOADERS FUNCIONANDO PERFECTAMENTE!")
        return True
        
    except Exception as e:
        print(f"❌ Error verificando DataLoaders: {e}")
        return False

def ejecutar_paso4_dataloaders(paths, mappings, device, batch_size=32):
    """Ejecutar paso 4 completo"""
    print("🚀 PASO 4: CREAR DATALOADERS")
    print("=" * 30)
    
    # Crear DataLoaders
    train_loader, val_loader, test_loader = crear_dataloaders(
        paths, mappings, batch_size, 'conservative'
    )
    
    # Verificar que funcionen
    dataloaders_ok = verificar_dataloaders(train_loader, val_loader, test_loader, device)
    
    if dataloaders_ok:
        print(f"\n🎯 ¡PASO 4 COMPLETADO!")
        return train_loader, val_loader, test_loader
    else:
        print(f"\n❌ PASO 4 FALLÓ")
        return None, None, None

# EJECUTAR PASO 4
print("🚀 EJECUTANDO PASO 4: DATALOADERS")
print("=" * 40)

batch_size = 32  # Ajusta según tu GPU

train_loader, val_loader, test_loader = ejecutar_paso4_dataloaders(
    paths, mappings, device, batch_size
)

if train_loader is not None:
    print(f"\n✅ DATALOADERS LISTOS!")
    print(f"📊 Configuración:")
    print(f"   Batch size: {batch_size}")
    print(f"   Transformaciones: conservative")
    print(f"   Device: {device}")
else:
    print(f"\n❌ ERROR EN DATALOADERS")

🚀 EJECUTANDO PASO 4: DATALOADERS
🚀 PASO 4: CREAR DATALOADERS
🚀 CREANDO DATALOADERS
🔧 Configurando transformaciones: conservative
📊 Creando datasets...
📊 Dataset creado: 100,000 imágenes
📊 Dataset creado: 6,000 imágenes
🧪 Test Dataset: 4,000 imágenes
⚙️ Configuración DataLoaders:
   Batch size: 32
   Num workers: 0
   Pin memory: True

📊 DataLoaders creados:
   Train: 100,000 imágenes, 3,125 batches
   Val: 6,000 imágenes, 188 batches
   Test: 4,000 imágenes, 125 batches

🔍 VERIFICANDO DATALOADERS
📏 Shapes:
   Train: torch.Size([32, 3, 64, 64]) (imágenes), torch.Size([32]) (labels)
   Val: torch.Size([32, 3, 64, 64]) (imágenes), torch.Size([32]) (labels)
   Test: torch.Size([32, 3, 64, 64]) (imágenes)

🔥 Probando transferencia a cuda...
   ✅ Transferencia exitosa

🎉 ¡DATALOADERS FUNCIONANDO PERFECTAMENTE!

🎯 ¡PASO 4 COMPLETADO!

✅ DATALOADERS LISTOS!
📊 Configuración:
   Batch size: 32
   Transformaciones: conservative
   Device: cuda


In [6]:
# ================================================================================
# BLOQUE 5: CREAR MODELO EFFICIENTNET-B4
# ================================================================================

def crear_efficientnet_b4(num_classes=200, pretrained=True):
    """Crear EfficientNet-B4 adaptado para Tiny ImageNet"""
    print("🤖 CREANDO EFFICIENTNET-B4")
    print("=" * 28)
    
    if pretrained:
        print("📥 Descargando pesos pre-entrenados de ImageNet...")
        model = models.efficientnet_b4(weights='IMAGENET1K_V1')
        print("   ✅ Pesos ImageNet cargados")
    else:
        print("🎲 Creando modelo desde cero...")
        model = models.efficientnet_b4(weights=None)
    
    # Adaptar classifier para Tiny ImageNet
    print(f"🔧 Adaptando classifier para {num_classes} clases...")
    
    in_features = model.classifier[1].in_features
    print(f"   Features de entrada: {in_features}")
    
    model.classifier[1] = nn.Linear(in_features, num_classes)
    print(f"   ✅ Classifier adaptado: {in_features} -> {num_classes}")
    
    return model

def obtener_info_modelo(model):
    """Obtener información detallada del modelo"""
    print("\n📊 INFORMACIÓN DEL MODELO")
    print("=" * 27)
    
    # Contar parámetros
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"📈 Parámetros:")
    print(f"   Total: {total_params:,} ({total_params/1e6:.1f}M)")
    print(f"   Entrenables: {trainable_params:,} ({trainable_params/1e6:.1f}M)")
    
    print(f"\n🏗️ Arquitectura:")
    print(f"   Modelo: EfficientNet-B4")
    print(f"   Input: (3, 64, 64) imágenes RGB")
    print(f"   Output: {model.classifier[1].out_features} clases")
    
    try:
        model_device = next(model.parameters()).device
        print(f"   Device: {model_device}")
    except:
        print(f"   Device: No inicializado")

def mover_modelo_a_device(model, device):
    """Mover modelo al device y verificar"""
    print(f"\n🔥 MOVIENDO MODELO A {device.type.upper()}")
    print("=" * 25)
    
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        memoria_antes = torch.cuda.memory_allocated() / 1024**3
        print(f"   Memoria GPU antes: {memoria_antes:.2f} GB")
    
    model = model.to(device)
    print(f"   ✅ Modelo movido a {device}")
    
    model_device = next(model.parameters()).device
    if model_device.type == device.type:
        print(f"   ✅ Verificación exitosa: modelo en {model_device}")
    else:
        print(f"   ❌ Error: modelo en {model_device}, esperado {device}")
        return None
    
    if device.type == 'cuda':
        memoria_despues = torch.cuda.memory_allocated() / 1024**3
        memoria_usada = memoria_despues - memoria_antes
        print(f"   Memoria GPU después: {memoria_despues:.2f} GB")
        print(f"   Memoria usada por modelo: {memoria_usada:.2f} GB")
    
    return model

def verificar_modelo(model, device, batch_size=8):
    """Verificar que el modelo funcione correctamente"""
    print(f"\n🔍 VERIFICANDO MODELO")
    print("=" * 20)
    
    model.eval()
    
    try:
        print("🧪 Test 1: Forward pass...")
        test_input = torch.randn(batch_size, 3, 64, 64).to(device)
        
        with torch.no_grad():
            output = model(test_input)
        
        print(f"   Input: {test_input.shape}")
        print(f"   Output: {output.shape}")
        print(f"   ✅ Forward pass exitoso")
        
        if output.shape == (batch_size, 200):
            print(f"   ✅ Output shape correcto: {output.shape}")
        else:
            print(f"   ❌ Output shape incorrecto: {output.shape}")
            return False
        
        # Test probabilidades
        probs = torch.softmax(output, dim=1)
        max_prob = probs.max().item()
        print(f"   Probabilidad máxima: {max_prob:.3f}")
        
        # Test predicciones
        predictions = torch.argmax(output, dim=1)
        print(f"   Predicciones sample: {predictions[:5].tolist()}")
        
        pred_min, pred_max = predictions.min().item(), predictions.max().item()
        if pred_min >= 0 and pred_max < 200:
            print(f"   ✅ Predicciones en rango válido [0-199]")
        else:
            print(f"   ❌ Predicciones fuera de rango: [{pred_min}, {pred_max}]")
            return False
        
        # Limpiar memoria
        del test_input, output, probs, predictions
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        
        print(f"   ✅ Modelo verificado correctamente")
        return True
        
    except Exception as e:
        print(f"   ❌ Error en verificación: {e}")
        return False

def test_modelo_con_dataloader(model, val_loader, device, num_batches=2):
    """Probar modelo con DataLoader real"""
    print(f"\n🚀 TEST CON DATALOADER REAL")
    print("=" * 28)
    
    model.eval()
    
    try:
        with torch.no_grad():
            for batch_idx, (images, labels) in enumerate(val_loader):
                if batch_idx >= num_batches:
                    break
                
                print(f"🔄 Batch {batch_idx + 1}:")
                
                images = images.to(device)
                labels = labels.to(device)
                
                print(f"   Input: {images.shape} en {images.device}")
                print(f"   Labels: {labels.shape} en {labels.device}")
                
                outputs = model(images)
                print(f"   Output: {outputs.shape}")
                
                _, predicted = torch.max(outputs, 1)
                correct = (predicted == labels).sum().item()
                accuracy = correct / labels.size(0) * 100
                
                print(f"   Accuracy batch: {accuracy:.1f}%")
                print(f"   Predicciones: {predicted[:5].tolist()}")
                print(f"   Labels reales: {labels[:5].tolist()}")
                
        print(f"   ✅ Test con DataLoader exitoso")
        return True
        
    except Exception as e:
        print(f"   ❌ Error con DataLoader: {e}")
        return False

def ejecutar_paso5_modelo(device, val_loader=None):
    """Ejecutar paso 5 completo"""
    print("🤖 PASO 5: CREAR MODELO EFFICIENTNET-B4")
    print("=" * 40)
    
    # Crear modelo
    model = crear_efficientnet_b4(num_classes=200, pretrained=True)
    
    # Mostrar información
    obtener_info_modelo(model)
    
    # Mover a device
    model = mover_modelo_a_device(model, device)
    
    if model is None:
        print("❌ Error moviendo modelo a device")
        return None
    
    # Verificar funcionamiento
    if not verificar_modelo(model, device):
        print("❌ Error en verificación del modelo")
        return None
    
    # Test con DataLoader si está disponible
    if val_loader is not None:
        if not test_modelo_con_dataloader(model, val_loader, device):
            print("❌ Error en test con DataLoader")
            return None
    
    print(f"\n🎯 ¡PASO 5 COMPLETADO!")
    print(f"✅ Modelo EfficientNet-B4 listo para entrenar")
    
    return model

def guardar_modelo_inicial(model, paths, filename="model_inicial.pth"):
    """Guardar estado inicial del modelo"""
    filepath = paths['checkpoints'] / filename
    torch.save(model.state_dict(), filepath)
    print(f"💾 Modelo inicial guardado: {filepath}")

# EJECUTAR PASO 5
print("🤖 EJECUTANDO PASO 5: MODELO")
print("=" * 35)

try:
    model = ejecutar_paso5_modelo(device, val_loader)
    
    if model is not None:
        guardar_modelo_inicial(model, paths)
        
        print(f"\n🚀 ¡MODELO LISTO!")
        print(f"📊 Configuración:")
        print(f"   Arquitectura: EfficientNet-B4")
        print(f"   Clases: 200")
        print(f"   Pre-trained: ImageNet")
        print(f"   Device: {device}")
        print(f"   Parámetros: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
        
    else:
        print(f"\n❌ ERROR CREANDO MODELO")
        
except Exception as e:
    print(f"❌ Error en Paso 5: {e}")
    model = None

🤖 EJECUTANDO PASO 5: MODELO
🤖 PASO 5: CREAR MODELO EFFICIENTNET-B4
🤖 CREANDO EFFICIENTNET-B4
📥 Descargando pesos pre-entrenados de ImageNet...
   ✅ Pesos ImageNet cargados
🔧 Adaptando classifier para 200 clases...
   Features de entrada: 1792
   ✅ Classifier adaptado: 1792 -> 200

📊 INFORMACIÓN DEL MODELO
📈 Parámetros:
   Total: 17,907,216 (17.9M)
   Entrenables: 17,907,216 (17.9M)

🏗️ Arquitectura:
   Modelo: EfficientNet-B4
   Input: (3, 64, 64) imágenes RGB
   Output: 200 clases
   Device: cpu

🔥 MOVIENDO MODELO A CUDA
   Memoria GPU antes: 0.01 GB
   ✅ Modelo movido a cuda
   ✅ Verificación exitosa: modelo en cuda:0
   Memoria GPU después: 0.08 GB
   Memoria usada por modelo: 0.07 GB

🔍 VERIFICANDO MODELO
🧪 Test 1: Forward pass...
   Input: torch.Size([8, 3, 64, 64])
   Output: torch.Size([8, 200])
   ✅ Forward pass exitoso
   ✅ Output shape correcto: torch.Size([8, 200])
   Probabilidad máxima: 0.100
   Predicciones sample: [62, 62, 62, 62, 62]
   ✅ Predicciones en rango válido [0

In [7]:
# ================================================================================
# BLOQUE 6: PIPELINE DE ENTRENAMIENTO
# ================================================================================

def configurar_entrenamiento():
    """Configuración de entrenamiento basada en tu experimento exitoso"""
    print("⚙️ CONFIGURACIÓN DE ENTRENAMIENTO")
    print("=" * 35)
    
    config = {
        'epochs': 8,
        'learning_rate': 0.0008,
        'batch_size': 32,
        'weight_decay': 0.01,
        'label_smoothing': 0.05,
        'patience': 4,
        'save_checkpoints': True,
        'save_best_model': True,
        'print_every': 200,
        'validate_every_epoch': True
    }
    
    print(f"📊 Configuración:")
    for key, value in config.items():
        print(f"   {key}: {value}")
    
    return config

def configurar_optimizer_y_loss(model, config):
    """Configurar optimizer, loss y scheduler"""
    print(f"\n🔧 CONFIGURANDO OPTIMIZER Y LOSS")
    print("=" * 35)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
    print(f"   Loss: CrossEntropyLoss + label_smoothing({config['label_smoothing']})")
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'], 
        weight_decay=config['weight_decay']
    )
    print(f"   Optimizer: AdamW(lr={config['learning_rate']}, weight_decay={config['weight_decay']})")
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',
        factor=0.5, 
        patience=3
    )
    print(f"   Scheduler: ReduceLROnPlateau")
    
    return criterion, optimizer, scheduler

def entrenar_epoca(model, train_loader, optimizer, criterion, device, config, epoch):
    """Entrenar una época completa"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    epoch_start = time.time()
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if batch_idx % config['print_every'] == 0:
            current_acc = 100 * correct / total if total > 0 else 0
            elapsed = time.time() - epoch_start
            progress = batch_idx / len(train_loader) * 100
            current_lr = optimizer.param_groups[0]['lr']
            
            print(f"  📈 Época {epoch+1} - Batch {batch_idx:4d}/{len(train_loader)} ({progress:.1f}%)")
            print(f"      Loss: {loss.item():.4f} | Acc: {current_acc:.1f}% | LR: {current_lr:.6f}")
            print(f"      Tiempo: {elapsed/60:.1f}min")
            
            if device.type == 'cuda':
                gpu_memory = torch.cuda.memory_allocated() / 1024**3
                print(f"      GPU: {gpu_memory:.2f}GB")
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    epoch_time = time.time() - epoch_start
    
    return epoch_loss, epoch_acc, epoch_time

def validar_epoca(model, val_loader, criterion, device):
    """Validar una época completa"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    val_start = time.time()
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total
    val_time = time.time() - val_start
    
    return val_loss, val_acc, val_time

def guardar_checkpoint(model, optimizer, epoch, train_loss, train_acc, val_loss, val_acc, config, checkpoints_dir, filename=None):
    """Guardar checkpoint completo"""
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        filename = f"checkpoint_epoch_{epoch+1}_{timestamp}.pth"
    
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'config': config,
        'timestamp': datetime.now().isoformat()
    }
    
    filepath = checkpoints_dir / filename
    torch.save(checkpoint, filepath)
    return filepath

def guardar_mejor_modelo(model, val_acc, checkpoints_dir, filename="best_model.pth"):
    """Guardar mejor modelo"""
    filepath = checkpoints_dir / filename
    torch.save(model.state_dict(), filepath)
    print(f"    💾 🏆 Mejor modelo guardado: {filepath} ({val_acc:.2f}%)")

def entrenar_modelo(model, train_loader, val_loader, device, config=None, checkpoints_dir=None):
    """Función principal de entrenamiento"""
    print("🚀 INICIANDO ENTRENAMIENTO")
    print("=" * 50)
    
    if config is None:
        config = configurar_entrenamiento()
    
    criterion, optimizer, scheduler = configurar_optimizer_y_loss(model, config)
    
    best_val_acc = 0.0
    patience_counter = 0
    
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    
    print(f"\n🎯 COMENZANDO ENTRENAMIENTO")
    print(f"📊 Dataset: {len(train_loader.dataset):,} train, {len(val_loader.dataset):,} val")
    print(f"⏱️ Tiempo estimado: ~{config['epochs'] * 15:.0f} minutos")
    
    start_time = time.time()
    
    for epoch in range(config['epochs']):
        print(f"\n{'='*70}")
        print(f"ÉPOCA {epoch+1}/{config['epochs']}")
        print(f"{'='*70}")
        
        # Entrenar época
        train_loss, train_acc, train_time = entrenar_epoca(
            model, train_loader, optimizer, criterion, device, config, epoch
        )
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        
        # Validar época
        print(f"\n📊 Validando...")
        val_loss, val_acc, val_time = validar_epoca(model, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        scheduler.step(val_acc)
        
        # Mostrar resultados
        total_time = time.time() - start_time
        
        print(f"\n📊 RESULTADOS ÉPOCA {epoch+1}:")
        print(f"   🚂 TRAIN - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"   ✅ VAL   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
        print(f"   📈 Gap (overfitting): {val_loss - train_loss:.3f}")
        print(f"   ⏱️ Tiempo época: {(train_time + val_time)/60:.1f} min")
        print(f"   ⏱️ Tiempo total: {total_time/60:.1f} min")
        
        # Análisis vs baseline
        baseline_acc = 58.0
        improvement = val_acc - baseline_acc
        print(f"   🎯 vs Baseline (58%): {improvement:+.2f}% ", end="")
        
        if improvement >= 2.0:
            print("(🔥 Excelente mejora!)")
        elif improvement >= 1.0:
            print("(✅ Buena mejora)")
        elif improvement >= 0.0:
            print("(📈 Mejora leve)")
        else:
            print("(⚠️ Por debajo baseline)")
        
        # Guardar mejor modelo
        improved = False
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            guardar_mejor_modelo(model, val_acc, checkpoints_dir)
            improved = True
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Guardar checkpoint
        if config['save_checkpoints']:
            checkpoint_file = guardar_checkpoint(
                model, optimizer, epoch, train_loss, train_acc, val_loss, val_acc, config, checkpoints_dir
            )
            print(f"    💾 Checkpoint: {checkpoint_file}")
        
        print(f"   🏆 Mejor accuracy hasta ahora: {best_val_acc:.2f}%")
        
        if not improved:
            print(f"   ⏳ Sin mejora - Paciencia: {patience_counter}/{config['patience']}")
        
        # Early stopping
        if patience_counter >= config['patience']:
            print(f"\n🛑 EARLY STOPPING - Sin mejora en {config['patience']} épocas")
            break
    
    # Resumen final
    total_time = time.time() - start_time
    
    print(f"\n🎉 ENTRENAMIENTO COMPLETADO!")
    print("=" * 50)
    print(f"⏱️ Tiempo total: {total_time/60:.1f} minutos ({total_time/3600:.2f} horas)")
    print(f"📊 Épocas completadas: {len(val_accuracies)}")
    print(f"🏆 MEJOR ACCURACY: {best_val_acc:.2f}%")
    print(f"🎯 Mejora vs baseline: {best_val_acc - baseline_acc:+.2f}%")
    
    # Análisis final
    target = 60.0
    if best_val_acc >= target:
        print(f"🎉 ¡TARGET 60% ALCANZADO!")
    else:
        print(f"⚠️ Target 60%: Faltan {target - best_val_acc:.2f}%")
    
    # Retornar historial
    history = {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'best_val_acc': best_val_acc,
        'total_time': total_time,
        'config': config
    }
    
    return history

# EJECUTAR ENTRENAMIENTO
print("🚀 PASO 6: ENTRENAMIENTO")
print("=" * 25)

if model is not None and train_loader is not None and val_loader is not None:
    print("✅ Todo listo para entrenar:")
    print(f"   Modelo: EfficientNet-B4 en {device}")
    print(f"   Train: {len(train_loader.dataset):,} imágenes")
    print(f"   Val: {len(val_loader.dataset):,} imágenes")
    
    config = configurar_entrenamiento()
    
    print(f"\n🚀 ¿LISTO PARA COMENZAR?")
    print(f"⏱️ Tiempo estimado: ~{config['epochs'] * 15} minutos")
    print(f"🎯 Objetivo: Superar 58% accuracy baseline")
    
    print(f"\n🔥 INICIANDO EN 3... 2... 1...")
    
    try:
        history = entrenar_modelo(model, train_loader, val_loader, device, config, checkpoints_dir=paths['checkpoints'])
        
        print(f"\n🎊 ¡ENTRENAMIENTO EXITOSO!")
        print(f"🏆 Mejor resultado: {history['best_val_acc']:.2f}%")
        
    except Exception as e:
        print(f"\n❌ Error durante entrenamiento: {e}")
        print("💡 Revisar memoria GPU o reducir batch_size")

else:
    print("❌ Faltan componentes para entrenar")

🚀 PASO 6: ENTRENAMIENTO
✅ Todo listo para entrenar:
   Modelo: EfficientNet-B4 en cuda
   Train: 100,000 imágenes
   Val: 6,000 imágenes
⚙️ CONFIGURACIÓN DE ENTRENAMIENTO
📊 Configuración:
   epochs: 8
   learning_rate: 0.0008
   batch_size: 32
   weight_decay: 0.01
   label_smoothing: 0.05
   patience: 4
   save_checkpoints: True
   save_best_model: True
   print_every: 200
   validate_every_epoch: True

🚀 ¿LISTO PARA COMENZAR?
⏱️ Tiempo estimado: ~120 minutos
🎯 Objetivo: Superar 58% accuracy baseline

🔥 INICIANDO EN 3... 2... 1...
🚀 INICIANDO ENTRENAMIENTO

🔧 CONFIGURANDO OPTIMIZER Y LOSS
   Loss: CrossEntropyLoss + label_smoothing(0.05)
   Optimizer: AdamW(lr=0.0008, weight_decay=0.01)
   Scheduler: ReduceLROnPlateau

🎯 COMENZANDO ENTRENAMIENTO
📊 Dataset: 100,000 train, 6,000 val
⏱️ Tiempo estimado: ~120 minutos

ÉPOCA 1/8
  📈 Época 1 - Batch    0/3125 (0.0%)
      Loss: 5.2739 | Acc: 6.2% | LR: 0.000800
      Tiempo: 0.0min
      GPU: 0.29GB
  📈 Época 1 - Batch  200/3125 (6.4%)
    

In [8]:
# ================================================================================
# BLOQUE 7: FINE-TUNING PROGRESIVO (OPCIONAL)
# ================================================================================

# ⚠️ EJECUTAR SOLO DESPUÉS DEL ENTRENAMIENTO BASE

def configurar_fine_tuning():
    """Configuración para fine-tuning progresivo"""
    fases_config = {
        'fase1': {
            'nombre': 'Classifier Only',
            'epochs': 3,
            'lr_classifier': 0.0002,
            'freeze_features': True,
            'patience': 2
        },
        'fase2': {
            'nombre': 'Partial Unfreeze', 
            'epochs': 4,
            'lr_classifier': 0.0001,
            'lr_features': 0.00005,
            'unfreeze_layers': 2,
            'patience': 3
        },
        'fase3': {
            'nombre': 'Full Fine-tuning',
            'epochs': 5,
            'lr_classifier': 0.00005,
            'lr_features': 0.00002,
            'unfreeze_all': True,
            'patience': 3
        }
    }
    return fases_config

def configurar_parametros_fase(model, config_fase):
    """Configurar qué parámetros entrenar en cada fase"""
    # Congelar todo primero
    for param in model.parameters():
        param.requires_grad = False
    
    # Siempre entrenar classifier
    for param in model.classifier.parameters():
        param.requires_grad = True
    
    trainable_params = sum(p.numel() for p in model.classifier.parameters())
    
    if config_fase.get('freeze_features'):
        pass  # Solo classifier
    elif config_fase.get('unfreeze_layers'):
        n_layers = config_fase['unfreeze_layers']
        features_layers = list(model.features.children())
        for layer in features_layers[-n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
                trainable_params += param.numel()
    elif config_fase.get('unfreeze_all'):
        for param in model.features.parameters():
            param.requires_grad = True
            trainable_params += param.numel()
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"   📊 Parámetros entrenables: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.1f}%)")
    
    return trainable_params

def crear_optimizer_fase(model, config_fase):
    """Crear optimizer con learning rates diferenciados"""
    if config_fase.get('freeze_features'):
        optimizer = optim.AdamW(
            model.classifier.parameters(), 
            lr=config_fase['lr_classifier'], 
            weight_decay=0.01
        )
    else:
        param_groups = [
            {'params': model.classifier.parameters(), 'lr': config_fase['lr_classifier']},
            {'params': model.features.parameters(), 'lr': config_fase['lr_features']}
        ]
        optimizer = optim.AdamW(param_groups, weight_decay=0.01)
    
    return optimizer

def entrenar_fase(model, train_loader, val_loader, device, config_fase, fase_nombre, checkpoints_dir):
    """Entrenar una fase específica del fine-tuning"""
    print(f"\n🚀 {fase_nombre.upper()}: {config_fase['nombre']}")
    print("=" * 50)
    
    configurar_parametros_fase(model, config_fase)
    optimizer = crear_optimizer_fase(model, config_fase)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss()
    
    fase_history = {
        'train_losses': [], 'train_accuracies': [],
        'val_losses': [], 'val_accuracies': [],
        'best_val_acc': 0.0
    }
    
    patience_counter = 0
    
    for epoch in range(config_fase['epochs']):
        print(f"\n{fase_nombre.upper()} - ÉPOCA {epoch+1}/{config_fase['epochs']}")
        
        # Entrenar
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if batch_idx % 300 == 0:
                current_acc = 100 * correct / total if total > 0 else 0
                print(f"    Batch {batch_idx:4d} - Loss: {loss.item():.4f} - Acc: {current_acc:.1f}%")
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        
        # Validar
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        
        fase_history['train_losses'].append(train_loss)
        fase_history['train_accuracies'].append(train_acc)
        fase_history['val_losses'].append(val_loss)
        fase_history['val_accuracies'].append(val_acc)
        
        scheduler.step(val_acc)
        
        print(f"   🚂 Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
        print(f"   ✅ Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
        
        if val_acc > fase_history['best_val_acc']:
            fase_history['best_val_acc'] = val_acc
            torch.save(model.state_dict(), checkpoints_dir / f'best_model_{fase_nombre.lower()}.pth')
            print(f"   💾 🏆 Mejor {fase_nombre}: {val_acc:.2f}%")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"   ⏳ Sin mejora: {patience_counter}/{config_fase['patience']}")
        
        if patience_counter >= config_fase['patience']:
            print(f"   🛑 Early stopping {fase_nombre}")
            break
    
    print(f"🎯 {fase_nombre.upper()} COMPLETADA: {fase_history['best_val_acc']:.2f}%")
    return fase_history

def ejecutar_fine_tuning_progresivo(model, train_loader, val_loader, device, paths):
    """Ejecutar fine-tuning progresivo completo en 3 fases"""
    print("🎯 FINE-TUNING PROGRESIVO COMPLETO")
    print("=" * 50)
    
    # Cargar modelo base entrenado
    modelo_base_path = paths['checkpoints'] / "best_model.pth"
    checkpoints_dir = paths['checkpoints']
    print(f"📥 Cargando modelo base: {modelo_base_path}")
    
    if modelo_base_path.exists():
        model.load_state_dict(torch.load(modelo_base_path, map_location=device))
        print("   ✅ Modelo base cargado exitosamente")
    else:
        print(f"   ❌ Archivo no encontrado: {modelo_base_path}")
        return None
    
    # Configurar fases
    fases_config = configurar_fine_tuning()
    historial_completo = {'fases': {}, 'best_overall_acc': 0.0}
    
    total_ft_start = time.time()
    
    # Ejecutar cada fase
    for fase_nombre, config_fase in fases_config.items():
        fase_history = entrenar_fase(model, train_loader, val_loader, device, config_fase, fase_nombre, checkpoints_dir)
        historial_completo['fases'][fase_nombre] = fase_history
        
        # Actualizar mejor accuracy general
        if fase_history['best_val_acc'] > historial_completo['best_overall_acc']:
            historial_completo['best_overall_acc'] = fase_history['best_val_acc']
            torch.save(model.state_dict(), checkpoints_dir / 'best_model_finetuned_final.pth')
            print(f"   🏆 NUEVO MEJOR MODELO GENERAL: {fase_history['best_val_acc']:.2f}%")
    
    total_ft_time = time.time() - total_ft_start
    
    # Resumen final
    print(f"\n🎉 FINE-TUNING PROGRESIVO COMPLETADO!")
    print("=" * 60)
    print(f"⏱️ Tiempo total fine-tuning: {total_ft_time/60:.1f} minutos")
    print(f"🏆 MEJOR ACCURACY FINAL: {historial_completo['best_overall_acc']:.2f}%")
    
    print(f"\n📊 Resumen por fases:")
    for fase_nombre, historia in historial_completo['fases'].items():
        print(f"   {fase_nombre.upper()}: {historia['best_val_acc']:.2f}%")
    
    baseline = 58.0
    mejora_total = historial_completo['best_overall_acc'] - baseline
    print(f"\n🎯 ANÁLISIS FINAL:")
    print(f"   Baseline original: {baseline}%")
    print(f"   Resultado final: {historial_completo['best_overall_acc']:.2f}%")
    print(f"   Mejora total: {mejora_total:+.2f}%")
    
    target = 60.0
    if historial_completo['best_overall_acc'] >= target:
        print(f"   🎉 ¡TARGET 60% ALCANZADO!")
    else:
        falta = target - historial_completo['best_overall_acc']
        print(f"   ⚠️ Target 60%: Faltan {falta:.2f}%")
    
    print(f"\n💾 Modelos guardados:")
    print(f"   🏆 best_model_finetuned_final.pth (MEJOR)")
    for fase in fases_config.keys():
        print(f"   📁 best_model_{fase}.pth")
    
    return historial_completo

# EJEMPLO DE USO (ejecutar solo DESPUÉS del entrenamiento base):
"""
print("🎯 PREPARANDO FINE-TUNING")
print("=" * 25)

if (paths['checkpoints'] / 'best_model.pth').exists():
    print("✅ Modelo base encontrado")
    print("🚀 Ejecutar fine-tuning:")
    print()
    print("historial_ft = ejecutar_fine_tuning_progresivo(model, train_loader, val_loader, device, paths)")
else:
    print("⏳ Esperando que termine entrenamiento base...")
"""

'\nprint("🎯 PREPARANDO FINE-TUNING")\nprint("=" * 25)\n\nif (paths[\'checkpoints\'] / \'best_model.pth\').exists():\n    print("✅ Modelo base encontrado")\n    print("🚀 Ejecutar fine-tuning:")\n    print()\n    print("historial_ft = ejecutar_fine_tuning_progresivo(model, train_loader, val_loader, device, paths)")\nelse:\n    print("⏳ Esperando que termine entrenamiento base...")\n'

In [9]:
# ================================================================================
# BLOQUE 8: GENERAR PREDICCIONES FINALES
# ================================================================================

def seleccionar_mejor_modelo(checkpoints_dir):
    """Seleccionar el mejor modelo disponible"""
    print("🏆 SELECCIONANDO MEJOR MODELO")
    print("=" * 30)
    
    modelos_disponibles = [
        'best_model_finetuned_final.pth',  # Mejor después de fine-tuning
        'best_model_fase3.pth',            # Mejor de fase 3
        'best_model_fase2.pth',            # Mejor de fase 2  
        'best_model_fase1.pth',            # Mejor de fase 1
        'best_model.pth',                  # Modelo base
    ]
    
    for modelo in modelos_disponibles:
        modelo_path = Path(checkpoints_dir) / modelo
        if modelo_path.exists():
            print(f"   ✅ Encontrado: {modelo_path}")
            
            size_mb = os.path.getsize(modelo_path) / 1024 / 1024
            mod_time = os.path.getmtime(modelo_path)
            mod_time_str = datetime.fromtimestamp(mod_time).strftime("%Y-%m-%d %H:%M")
            
            print(f"      📊 Tamaño: {size_mb:.1f} MB")
            print(f"      📅 Modificado: {mod_time_str}")
            print(f"   🎯 USANDO ESTE MODELO")
            
            return modelo_path
    
    print("   ❌ No se encontró ningún modelo")
    return None

def cargar_modelo_para_predicciones(model, modelo_path, device):
    """Cargar el modelo seleccionado"""
    print(f"\n📥 CARGANDO MODELO: {modelo_path}")
    print("=" * 40)
    
    try:
        model.load_state_dict(torch.load(modelo_path, map_location=device))
        model.eval()
        
        print("   ✅ Modelo cargado exitosamente")
        
        # Verificar que funciona
        test_input = torch.randn(1, 3, 64, 64).to(device)
        with torch.no_grad():
            test_output = model(test_input)
        
        print(f"   ✅ Test exitoso: {test_output.shape}")
        
        del test_input, test_output
        torch.cuda.empty_cache()
        
        return True
        
    except Exception as e:
        print(f"   ❌ Error cargando modelo: {e}")
        return False

def generar_predicciones(model, test_loader, device):
    """Generar predicciones en el test set"""
    print(f"\n🔮 GENERANDO PREDICCIONES")
    print("=" * 30)
    
    model.eval()
    
    predicciones = []
    filenames = []
    
    print(f"📊 Test set: {len(test_loader.dataset):,} imágenes")
    print(f"🚀 Iniciando inferencia...")
    
    start_time = time.time()
    
    with torch.no_grad():
        for batch_idx, (images, names) in enumerate(test_loader):
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            predicciones.extend(predicted.cpu().numpy().tolist())
            filenames.extend(names)
            
            if batch_idx % 20 == 0 or batch_idx == len(test_loader) - 1:
                progress = (batch_idx + 1) / len(test_loader) * 100
                elapsed = time.time() - start_time
                print(f"   📈 Progreso: {progress:.1f}% ({len(predicciones):,} predicciones) - {elapsed:.1f}s")
    
    total_time = time.time() - start_time
    
    print(f"\n✅ PREDICCIONES COMPLETADAS")
    print(f"   Total imágenes: {len(predicciones):,}")
    print(f"   Tiempo total: {total_time:.1f} segundos")
    print(f"   Velocidad: {len(predicciones)/total_time:.1f} imágenes/segundo")
    
    return predicciones, filenames

def crear_submission_file(predicciones, filenames, modelo_usado):
    """Crear archivo de submission en formato correcto"""
    print(f"\n📄 CREANDO ARCHIVO DE SUBMISSION")
    print("=" * 35)
    
    submission_df = pd.DataFrame({
        'id': filenames,
        'pred': predicciones
    })
    
    print(f"📊 Verificando formato...")
    print(f"   Columnas: {list(submission_df.columns)}")
    print(f"   Filas: {len(submission_df):,}")
    
    # Verificaciones
    assert list(submission_df.columns) == ['id', 'pred'], "❌ Columnas incorrectas"
    assert len(predicciones) > 0, "❌ No hay predicciones"
    assert min(predicciones) >= 0 and max(predicciones) < 200, "❌ Predicciones fuera de rango"
    
    print(f"   ✅ Formato correcto")
    
    # Estadísticas
    unique_classes = len(set(predicciones))
    print(f"\n📊 Estadísticas de predicciones:")
    print(f"   Clases únicas predichas: {unique_classes}/200")
    print(f"   Rango: {min(predicciones)} - {max(predicciones)}")
    
    pred_counts = pd.Series(predicciones).value_counts().head(5)
    print(f"   Top 5 clases predichas:")
    for clase, count in pred_counts.items():
        print(f"      Clase {clase}: {count} imágenes ({count/len(predicciones)*100:.1f}%)")
    
    # Crear nombre de archivo
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    modelo_name = Path(modelo_usado).name.replace('.pth', '').replace('best_model_', '')
    
    filename = f"submission_{modelo_name}_{timestamp}.csv"
    
    # Guardar archivo
    submission_df.to_csv(filename, index=False)
    
    print(f"\n💾 ARCHIVO GUARDADO: {filename}")
    
    # Verificar
    if os.path.exists(filename):
        file_size = os.path.getsize(filename) / 1024
        print(f"   ✅ Archivo verificado: {file_size:.1f} KB")
        
        verification_df = pd.read_csv(filename)
        if len(verification_df) == len(submission_df):
            print(f"   ✅ Verificación exitosa: {len(verification_df):,} filas")
        else:
            print(f"   ❌ Error en verificación")
    
    return filename

def mostrar_ejemplos_predicciones(filenames, predicciones, num_ejemplos=5):
    """Mostrar algunos ejemplos de predicciones"""
    print(f"\n📋 EJEMPLOS DE PREDICCIONES:")
    print("=" * 30)
    
    for i in range(min(num_ejemplos, len(filenames))):
        filename = filenames[i]
        pred = predicciones[i]
        print(f"   {filename} → Clase {pred}")

def ejecutar_predicciones_finales(model, test_loader, device, checkpoints_dir):
    """Pipeline completo para generar predicciones finales"""
    print("🔮 PIPELINE DE PREDICCIONES FINALES")
    print("=" * 50)
    
    # Seleccionar mejor modelo
    mejor_modelo = seleccionar_mejor_modelo(checkpoints_dir)
    
    if mejor_modelo is None:
        print("❌ No se encontró modelo para usar")
        return None
    
    # Cargar modelo
    if not cargar_modelo_para_predicciones(model, mejor_modelo, device):
        print("❌ Error cargando modelo")
        return None
    
    # Generar predicciones
    predicciones, filenames = generar_predicciones(model, test_loader, device)
    
    # Mostrar ejemplos
    mostrar_ejemplos_predicciones(filenames, predicciones)
    
    # Crear archivo de submission
    submission_file = crear_submission_file(predicciones, filenames, mejor_modelo)
    
    # Resumen final
    print(f"\n🎉 PREDICCIONES FINALES COMPLETADAS!")
    print("=" * 40)
    print(f"🏆 Modelo usado: {mejor_modelo}")
    print(f"📊 Predicciones: {len(predicciones):,} imágenes")
    print(f"📄 Archivo: {submission_file}")
    print(f"🎯 Listo para competición!")
    
    return submission_file

def verificar_submission_final(submission_file, paths):
    """Verificación final del archivo de submission"""
    print(f"\n🔍 VERIFICACIÓN FINAL DEL SUBMISSION")
    print("=" * 40)
    
    if not os.path.exists(submission_file):
        print(f"❌ Archivo no encontrado: {submission_file}")
        return False
    
    submission_df = pd.read_csv(submission_file)
    test_df = pd.read_csv(paths['test_csv'])
    
    print(f"📊 Comparando con test original...")
    
    checks = {
        'Formato columnas': list(submission_df.columns) == ['id', 'pred'],
        'Número de filas': len(submission_df) == len(test_df),
        'Sin duplicados': len(submission_df['id'].unique()) == len(submission_df),
        'Rango predicciones': (submission_df['pred'].min() >= 0) and (submission_df['pred'].max() < 200),
        'Sin valores nulos': submission_df.isnull().sum().sum() == 0
    }
    
    print(f"✅ Verificaciones:")
    all_passed = True
    for check_name, passed in checks.items():
        status = "✅" if passed else "❌"
        print(f"   {status} {check_name}")
        if not passed:
            all_passed = False
    
    if all_passed:
        print(f"\n🎉 ¡SUBMISSION PERFECTO!")
        print(f"🚀 Listo para subir a la competición")
        return True
    else:
        print(f"\n❌ Hay problemas en el submission")
        return False

# EJECUTAR PREDICCIONES FINALES
print("🔮 EJECUTANDO PREDICCIONES FINALES")
print("=" * 50)

if 'model' in locals() and 'test_loader' in locals():
    print("✅ Componentes listos:")
    print(f"   Modelo: Disponible")
    print(f"   Test loader: {len(test_loader.dataset):,} imágenes")
    print(f"   Device: {device}")
    
    submission_file = ejecutar_predicciones_finales(model, test_loader, device, checkpoints_dir=paths['checkpoints'])
    
    if submission_file:
        verificacion_ok = verificar_submission_final(submission_file, paths)
        
        if verificacion_ok:
            print(f"\n🏆 ¡TODO COMPLETADO EXITOSAMENTE!")
            print(f"📁 Archivo final: {submission_file}")
            print(f"🎯 Modelo entrenado y predicciones listas")
            print(f"🚀 ¡A competir!")
        else:
            print(f"\n⚠️ Submission creado pero con advertencias")
    
else:
    print("❌ Faltan componentes:")
    print("   Ejecuta primero los bloques anteriores")

🔮 EJECUTANDO PREDICCIONES FINALES
✅ Componentes listos:
   Modelo: Disponible
   Test loader: 4,000 imágenes
   Device: cuda
🔮 PIPELINE DE PREDICCIONES FINALES
🏆 SELECCIONANDO MEJOR MODELO
   ✅ Encontrado: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\checkpoints\best_model.pth
      📊 Tamaño: 69.0 MB
      📅 Modificado: 2026-03-19 01:08
   🎯 USANDO ESTE MODELO

📥 CARGANDO MODELO: C:\Users\david\Documents\Programming projects\Image classification\Image-Classification-Tiny-Imagenet\checkpoints\best_model.pth
   ✅ Modelo cargado exitosamente
   ✅ Test exitoso: torch.Size([1, 200])

🔮 GENERANDO PREDICCIONES
📊 Test set: 4,000 imágenes
🚀 Iniciando inferencia...
   📈 Progreso: 0.8% (32 predicciones) - 0.3s
   📈 Progreso: 16.8% (672 predicciones) - 16.4s
   📈 Progreso: 32.8% (1,312 predicciones) - 32.2s
   📈 Progreso: 48.8% (1,952 predicciones) - 47.0s
   📈 Progreso: 64.8% (2,592 predicciones) - 62.5s
   📈 Progreso: 80.8% (3,232 predicci